NOTE: outputs excluded

# Import Libraries

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from datetime import datetime
from collections import OrderedDict
from tqdm import tqdm

# Load Data

In [ ]:
import pandas as pd
import glob
import os
from scipy.interpolate import interp1d
import numpy as np

# Function to generate new ID
def generate_new_id(file_name):
    parts = file_name.split('_')
    participant_id = parts[0]
    video_id = parts[2].split('.')[0][-1]
    new_id = f"{participant_id}_{video_id}"
    return new_id

# Function to read and prepare BVP data
def read_bvp_data(base_dir, participant_id):
    participant_dir = os.path.join(base_dir, f'{participant_id:04d}', 'raw', 'empatica')
    bvp_files = glob.glob(os.path.join(participant_dir, '*bvp*'))
    df_list = []
    for file in bvp_files:
        df = pd.read_csv(file)
        df['participant_id'] = participant_id
        df['file_name'] = os.path.basename(file)
        df['participant_video'] = df['file_name'].apply(generate_new_id)
        df['bvp_datetime'] = pd.to_datetime(df['bvp_datetime'])
        df_list.append(df)
    return pd.concat(df_list, ignore_index=True)

# Function to read and prepare EDA data
def read_eda_data(base_dir, participant_id):
    participant_dir = os.path.join(base_dir, f'{participant_id:04d}', 'raw', 'empatica')
    eda_files = glob.glob(os.path.join(participant_dir, '*eda*'))
    df_list = []
    for file in eda_files:
        df = pd.read_csv(file)
        df['participant_id'] = participant_id
        df['file_name'] = os.path.basename(file)
        df['participant_video'] = df['file_name'].apply(generate_new_id)
        df['eda_datetime'] = pd.to_datetime(df['eda_datetime'])
        df_list.append(df)
    eda_df = pd.concat(df_list, ignore_index=True)
    eda_df.set_index('eda_datetime', inplace=True)
    return eda_df

# Function to read and prepare Systolic Peaks data
def read_systolic_peaks_data(base_dir, participant_id):
    participant_dir = os.path.join(base_dir, f'{participant_id:04d}', 'raw', 'empatica')
    systolic_files = glob.glob(os.path.join(participant_dir, '*systolicPeaks*'))
    df_list = []
    for file in systolic_files:
        df = pd.read_csv(file)
        df['participant_id'] = participant_id
        df['file_name'] = os.path.basename(file)
        df['participant_video'] = df['file_name'].apply(generate_new_id)
        df['systolic_datetime'] = pd.to_datetime(df['systolic_datetime'])
        df_list.append(df)
    return pd.concat(df_list, ignore_index=True)

# Function to calculate heart rate from systolic peaks data
def calculate_heart_rate(systolic_datetimes):
    rr_intervals = systolic_datetimes.diff().dt.total_seconds().dropna()
    heart_rates = 60 / rr_intervals
    return heart_rates

# Base directory
base_dir = r'C:\Users\roysu\OneDrive - Deakin University\DSSN2023_shared\participants'

# Initialize an empty list to hold DataFrames
combined_df_list = []

# Loop over the participant directories
for participant_id in range(10, 40):
    # Read and prepare data
    bvp_data = read_bvp_data(base_dir, participant_id)
    bvp_data.set_index('bvp_datetime', inplace=True)

    eda_data = read_eda_data(base_dir, participant_id)
    systolic_peaks_data = read_systolic_peaks_data(base_dir, participant_id)

    # Get unique participant_video combinations
    participant_videos = bvp_data['participant_video'].unique()

    # Merge data for each participant_video separately
    for participant_video in participant_videos:
        bvp_subset = bvp_data[bvp_data['participant_video'] == participant_video]
        eda_subset = eda_data[eda_data['participant_video'] == participant_video]
        systolic_peaks_subset = systolic_peaks_data[systolic_peaks_data['participant_video'] == participant_video].copy()

        # Convert DatetimeIndex to numeric (seconds relative to the first EDA timestamp)
        eda_first_time = eda_subset.index.min()
        eda_time_numeric = (eda_subset.index - eda_first_time).total_seconds()
        bvp_time_numeric = (bvp_subset.index - eda_first_time).total_seconds()

        # Upsample EDA to match BVP's 64 Hz using interpolation
        eda_values = eda_subset['eda_values']  # Assuming 'eda_values' is the EDA column name

        # Inside the participant_video loop, before interpolation
        eda_first = eda_subset.index.min()
        eda_last = eda_subset.index.max()
        bvp_first = bvp_subset.index.min()
        bvp_last = bvp_subset.index.max()

        extrap_before_seconds = (bvp_first - eda_first).total_seconds() if bvp_first < eda_first else 0
        extrap_after_seconds = (bvp_last - eda_last).total_seconds() if bvp_last > eda_last else 0
        extrap_before_samples = int(extrap_before_seconds / (1/64)) if extrap_before_seconds > 0 else 0
        extrap_after_samples = int(extrap_after_seconds / (1/64)) if extrap_after_seconds > 0 else 0

        # Measure BVP sampling interval
        bvp_intervals = bvp_subset.index.to_series().diff().dt.total_seconds() * 1000  # Convert to milliseconds
        avg_interval = bvp_intervals.mean()  # Average interval in ms
        std_interval = bvp_intervals.std()  # Standard deviation in ms
        intended_interval = 1000 / 64  # Intended interval at 64 Hz = 15.625ms
        offset_ms = avg_interval - intended_interval  # Offset from intended in ms
        offset_percent = (offset_ms / intended_interval) * 100  # Offset as percentage

        # Print BVP sampling analysis
        print(f"Participant {participant_id}, Video {participant_video}:")
        print(f"EDA Range: {eda_first} to {eda_last}")
        print(f"BVP Range: {bvp_first} to {bvp_last}")
        print(f"Extrapolation Before: {extrap_before_seconds:.2f}s ({extrap_before_samples} samples)")
        print(f"Extrapolation After: {extrap_after_seconds:.2f}s ({extrap_after_samples} samples)")
        print(f"Total Extrapolation: {(extrap_before_seconds + extrap_after_seconds):.2f}s ({extrap_before_samples + extrap_after_samples} samples)")
        print(f"BVP Avg Interval: {avg_interval:.3f}ms, Std Dev: {std_interval:.3f}ms")
        print(f"Intended Interval: {intended_interval:.3f}ms, Offset: {offset_ms:.3f}ms ({offset_percent:.2f}%)")

        # Upsample EDA to match BVP's 64 Hz using interpolation
        interp_func = interp1d(eda_time_numeric, eda_values, kind='linear', fill_value='extrapolate')
        eda_upsampled = pd.Series(interp_func(bvp_time_numeric), index=bvp_subset.index, name='eda_values')
        merged_data = bvp_subset.copy()
        merged_data['eda_values'] = eda_upsampled

        # Include the bvp and eda timestamps in the DataFrame
        merged_data['bvp_datetime'] = merged_data.index
        merged_data['eda_datetime'] = merged_data.index  # Since EDA is now aligned to BVP index

        # Adding systolic peaks as a flag if a systolic peak is near the BVP timestamp
        systolic_peaks_subset['systolic_peak_flag'] = 1
        merged_data = pd.merge_asof(merged_data.sort_index(),
                                    systolic_peaks_subset[['systolic_datetime', 'systolic_peak_flag']].sort_index(),
                                    left_index=True,
                                    right_on='systolic_datetime',
                                    tolerance=pd.Timedelta(milliseconds=10),
                                    direction='nearest')

        merged_data['systolic_peak_flag'].fillna(0, inplace=True)

        # Replace consecutive systolic peaks with 0
        merged_data['systolic_peak_flag_shift'] = merged_data['systolic_peak_flag'].shift().fillna(0)
        merged_data.loc[(merged_data['systolic_peak_flag'] == 1) & (merged_data['systolic_peak_flag_shift'] == 1), 'systolic_peak_flag'] = 0
        merged_data.drop(columns=['systolic_peak_flag_shift'], inplace=True)

        # Calculate HR for each interval between systolic peaks
        systolic_peaks_subset = merged_data[merged_data['systolic_peak_flag'] == 1]
        heart_rates = calculate_heart_rate(systolic_peaks_subset['systolic_datetime'])
        systolic_peaks_subset = systolic_peaks_subset.iloc[1:].copy()
        systolic_peaks_subset['heart_rate'] = heart_rates.values

        # Merge the heart rate data back into the main DataFrame with interpolation
        hr_time_numeric = (systolic_peaks_subset['systolic_datetime'] - eda_first_time).dt.total_seconds()
        hr_values = systolic_peaks_subset['heart_rate']
        interp_func_hr = interp1d(hr_time_numeric, hr_values, kind='linear', fill_value=np.nan, bounds_error=False)
        merged_data['heart_rate'] = interp_func_hr(bvp_time_numeric)

        # Apply ffill and bfill to handle start and end NaNs
        merged_data['heart_rate'] = merged_data['heart_rate'].fillna(method='ffill').fillna(method='bfill')

        # Append to the combined list
        combined_df_list.append(merged_data)

# Concatenate all the combined DataFrames into one
final_combined_df = pd.concat(combined_df_list, ignore_index=True)

# Filter out participants 29 and 35
participants_to_exclude = [29, 35]
filtered_df = final_combined_df[~final_combined_df['participant_id'].isin(participants_to_exclude)]

# Save the combined DataFrame to a CSV file
filtered_df.to_csv('final_combined_data_with_hr.csv', index=False)

# Remove leading zeros from participant IDs in the participant_video column
filtered_df['participant_video'] = filtered_df['participant_video'].apply(lambda x: f"{int(x.split('_')[0])}_{x.split('_')[1]}")

# Display the combined DataFrame
filtered_df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = filtered_df

# Convert bvp_datetime to datetime for indexing
df['bvp_datetime'] = pd.to_datetime(df['bvp_datetime'], utc=True)

# Get unique participant_videos
participant_videos = df['participant_video'].unique()

# Set up the plot
n_videos = len(participant_videos)
fig, axes = plt.subplots(nrows=(n_videos + 1) // 2, ncols=2, figsize=(15, 5 * ((n_videos + 1) // 2)), squeeze=False)
axes = axes.flatten()

# Plot heart rate and systolic peaks for each participant_video
for idx, video in enumerate(participant_videos):
    video_df = df[df['participant_video'] == video].set_index('bvp_datetime')
    time_numeric = (video_df.index - video_df.index.min()).total_seconds()

    # Plot heart rate
    axes[idx].plot(time_numeric, video_df['heart_rate'], label=f'HR for {video} (BPM)', color='blue', alpha=0.5)

    # Plot systolic peaks as dots (where systolic_peak_flag is 1)
    peak_indices = video_df.index[video_df['systolic_peak_flag'] == 1]
    peak_times = (peak_indices - video_df.index.min()).total_seconds()
    peak_hr_values = video_df.loc[peak_indices, 'heart_rate']  # HR at peak times
    axes[idx].scatter(peak_times, peak_hr_values, color='green', label='Systolic Peaks', s=10)

    axes[idx].set_xlabel('Time (seconds)')
    axes[idx].set_ylabel('Heart Rate (BPM)')
    axes[idx].set_title(f'Heart Rate for {video}')
    axes[idx].grid(True)

    # Highlight NaN regions
    nan_mask = video_df['heart_rate'].isna()
    axes[idx].fill_between(time_numeric, 0, video_df['heart_rate'].max() * 1.1, where=nan_mask, color='red', alpha=0.3, label='Missing Data (NaN)')
    axes[idx].legend()

# Remove empty subplots if any
for idx in range(len(participant_videos), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

# Optional: Display summary of NaNs per video
print("NaN counts per participant_video:")
print(df.groupby('participant_video')['heart_rate'].apply(lambda x: x.isna().sum()))

# Load Labels

In [ ]:
excel_file_path = r'DSSN2023/participants/surveys_notes/post_experience_survey.xlsx'
survey_df = pd.read_excel(excel_file_path)
# Filter the DataFrame to include only rows where 'EXCL' column is NaN
survey_df = survey_df[survey_df['EXCL'].isna()]
# Convert participant to integer
survey_df['Participant #'] = survey_df['Participant #'].astype(int)

# Convert video # to float
survey_df['Video #'] = survey_df['Video #'].astype(int)
survey_df['participant_video'] = survey_df['Participant #'].astype(str) + "_" + survey_df['Video #'].astype(str)
# List of columns to be dropped
columns_to_drop = ["Duration (in seconds)", "StartDate", "EndDate",
                   "RecordedDate", "DO", "M", "EXCL"]

# Drop the specified columns
survey_df.drop(columns_to_drop, axis=1, inplace=True)
survey_df

In [ ]:
merged_df = pd.merge(filtered_df, survey_df, on='participant_video', how='inner')
merged_df[['subject', 'video']] = merged_df['participant_video'].str.split('_', expand=True)
columns_to_drop = ['participant_video', 'participant']
merged_df = merged_df.drop(columns=columns_to_drop)
merged_df['participant_video'] = merged_df['Participant #'].astype(str) + "_" + merged_df['Video #'].astype(str)

In [ ]:
df = merged_df
# Check if 'bvp_datetime' and 'systolic_datetime' columns exist
if 'bvp_datetime' in merged_df_shuffled.columns and 'systolic_datetime' in merged_df_shuffled.columns:
    # Convert to datetime objects if they aren't already
    merged_df_shuffled['bvp_datetime'] = pd.to_datetime(merged_df_shuffled['bvp_datetime'])
    merged_df_shuffled['systolic_datetime'] = pd.to_datetime(merged_df_shuffled['systolic_datetime'])

    # Compare the columns and create a new boolean column
    merged_df_shuffled['datetimes_are_same'] = merged_df_shuffled['bvp_datetime'] == merged_df_shuffled['systolic_datetime']

    # Print or analyze the results
    print(merged_df_shuffled['datetimes_are_same'].value_counts())  # Counts of True/False
    # or
    print(merged_df_shuffled[merged_df_shuffled['datetimes_are_same'] == False]) # Rows where datetimes are not equal
else:
    print("Error: 'bvp_datetime' or 'systolic_datetime' column not found in the DataFrame.")

In [ ]:
columns_to_keep = ['VA', 'AR', 'subject', 'video', 'bvp_values', 'eda_values', 'heart_rate', 'bvp_datetime', 'eda_datetime', 'systolic_datetime']
df = df[columns_to_keep]
df['participant_video'] = df['subject'].astype(str) + "_" + df['video'].astype(str)
df

In [ ]:
df['systolic_datetime'] = df['systolic_datetime'].astype(str).str.replace(r'\+\d{2}:\d{2}$', '', regex=True)
df['systolic_datetime'] = pd.to_datetime(df['systolic_datetime'])
df

In [ ]:
# Rename columns
df = df.rename(columns={'subject': 'ID'})
df = df.rename(columns={'video': 'Trial'})
df = df.rename(columns={'systolic_datetime': 'timestamp'})

df['AR'] = pd.to_numeric(df['AR'], errors='coerce')
df['AR_Rating'] = df['AR'].apply(lambda x: 1 if x >= 5 else 0)
df['AR_Rating'] = pd.to_numeric(df['AR_Rating'])

df['VA'] = pd.to_numeric(df['VA'], errors='coerce')
df['VA_Rating'] = df['VA'].apply(lambda x: 1 if x >= 5 else 0)
df['VA_Rating'] = pd.to_numeric(df['VA_Rating'])
df = df.reset_index(drop=True)

#df = df.rename(columns={'AR_Rating_1': 'AR_Rating', 'VA_Rating_1': 'VA_Rating'})

df = df.sort_values(['ID', 'Trial']) # FOR MTL, shouldnt be ordered for population STL
df['ID_Trial'] = df['ID'].astype(str) + '_' + df['Trial'].astype(str)

from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the data
df['participant_trial_encoded'] = label_encoder.fit_transform(df['ID_Trial'])
df

In [ ]:
# Count the number of occurrences of each unique value in the 'VA_Rating' column
va_rating_counts = df['VA_Rating'].value_counts()

# Calculate the percentage of '0' values in the 'VA_Rating' column
percentage_zeros = (va_rating_counts[0] / df['VA_Rating'].count()) * 100

print(va_rating_counts)
print(percentage_zeros)

# Count the number of occurrences of each unique value in the 'VA_Rating' column
ar_rating_counts = df['AR_Rating'].value_counts()

# Calculate the percentage of '0' values in the 'VA_Rating' column
percentage_zeros = (ar_rating_counts[0] / df['AR_Rating'].count()) * 100

print(ar_rating_counts)
print(percentage_zeros)

VA_Rating
1    2883861
0     701696
Name: count, dtype: int64
19.570069587514578
AR_Rating
1    2940309
0     645248
Name: count, dtype: int64
17.995753518909336


In [ ]:
columns_to_drop = ['participant_video', 'video', 'VA', 'AR', 'Video #', 'subject']
df = df.drop(columns=columns_to_drop, errors='ignore')

# Reorder columns
new_column_order = ['ID', 'Trial', 'ID_Trial', 'participant_trial_encoded', 'eda_values', 'bvp_values', 'heart_rate', 'AR_Rating', 'VA_Rating']
df = df[new_column_order]
df

# Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

# Function to scale columns 'GSR' and 'ECG' within each group
def scale_columns(group):
    scaler = StandardScaler()
    group[['eda_values', 'bvp_values', 'heart_rate']] = scaler.fit_transform(group[['eda_values', 'bvp_values', 'heart_rate']])
    return group

# Apply the scaling function to each group by 'subject_video_encoded'
df = df.groupby('ID').apply(scale_columns)
df = df.reset_index(drop=True)
df

# Final Export for modeling

In [ ]:
df.to_csv('/content/drive/MyDrive/Phase A/data/DSSN_EM_data_v2.csv')

# END